[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PacktPublishing/Data-Strategy-for-LLMs/blob/main/chapter_07/Jupyter_Notebooks/Chapter_7_Notebook.ipynb)

**Click the badge above to run this notebook in Google Colab (no local setup needed).**


## Setup

**This chapter uses the book-wide shared environment (see the repository README).**

1. Run the book-wide setup once from the repo root: `bash setup/setup_mac.sh` (macOS/Linux) or `powershell -ExecutionPolicy Bypass -File setup/setup_windows.ps1` (Windows). This creates the `data_strategy_env` environment, the **"Python (Data Strategy Book)"** Jupyter kernel, and your API key.
2. Select the **"Python (Data Strategy Book)"** kernel (top-right). If missing: Command Palette -> "Developer: Reload Window".

The next cell installs any missing packages **into the running kernel** and loads your OpenAI API key (it prompts you if no `.env` key is found, e.g., on Colab).


In [ ]:
import warnings; warnings.filterwarnings("ignore")
import sys, subprocess
def _install(pkg):
    for cmd in ([sys.executable,"-m","pip","install",pkg,"--quiet"],
                [sys.executable,"-m","pip","install",pkg,"--user","--quiet"],
                [sys.executable,"-m","pip","install",pkg,"--break-system-packages","--quiet"]):
        try:
            subprocess.run(cmd, check=True, capture_output=True, text=True); return True
        except subprocess.CalledProcessError:
            continue
    return False
if not _install("openai"):
    print("WARNING: could not install openai (restart the kernel and re-run this cell)")

from openai import OpenAI

import sys, os
from pathlib import Path
_rr = Path.cwd()
for _p in [Path.cwd()] + list(Path.cwd().parents):
    if (_p / "utils" / "config.py").exists():
        _rr = _p; break
if str(_rr) not in sys.path:
    sys.path.insert(0, str(_rr))
try:
    from utils.config import get_openai_api_key
    api_key = get_openai_api_key()
except Exception:
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        import getpass
        api_key = getpass.getpass("Enter your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = api_key

client = OpenAI(api_key=api_key)

# Discover the best available model for SFT base (self-updating, cached)
from utils.models import get_best_available_model
BASE_MODEL = os.getenv("BASE_MODEL") or get_best_available_model(client)
print(f"SFT base model: {BASE_MODEL}")
print("Setup complete.")

Setup complete.


## Applying Supervised Fine-Tuning

Read minimal, "failure-shaped" SFT examples with stable structure and explicit uncertainty handling, upload them and launch a supervised fine-tuning job.

In [2]:
import json
from pathlib import Path

# Load pre-stored SFT datasets (10 train, 3 validation examples)
train_path = Path("../datasets/sft_train.jsonl")
valid_path = Path("../datasets/sft_valid.jsonl")

# Upload files for fine-tuning
with train_path.open("rb") as f:
    train_file = client.files.create(file=f, purpose="fine-tune")
with valid_path.open("rb") as f:
    valid_file = client.files.create(file=f, purpose="fine-tune")

# Create SFT job (keep hyperparameters default unless you have a reason)
job = client.fine_tuning.jobs.create(
    model=BASE_MODEL,
    training_file=train_file.id,
    validation_file=valid_file.id,
    method={"type": "supervised"},
)

print("SFT job:", job.id, "| status:", job.status)

PermissionDeniedError: Error code: 403 - {'error': {'message': 'OpenAI is winding down the fine-tuning platform and your organization is no longer able to create new fine-tuning training jobs. Learn more https://developers.openai.com/api/docs/deprecations#update-to-openais-self-serve-fine-tuning', 'type': 'invalid_request_error', 'param': None, 'code': 'training_not_available'}}

In [ ]:
# === Wait for SFT job to finish and save the fine-tuned model ID ===
# This cell polls the job until it completes, then saves the model name
# so Chapter 9 can load it for evaluation.

import time

job_id = job.id
print(f"Polling job {job_id}...")

while True:
    status = client.fine_tuning.jobs.retrieve(job_id)
    print(f"  Status: {status.status}")
    if status.status in ("succeeded", "failed", "cancelled"):
        break
    time.sleep(30)  # check every 30 seconds

if status.status == "succeeded":
    sft_model_name = status.fine_tuned_model
    print(f"\nFine-tuned model: {sft_model_name}")

    # Save the model ID so Chapter 9 can pick it up
    output_path = Path("../datasets/sft_model_id.txt")
    output_path.write_text(sft_model_name)
    print(f"Saved to {output_path}")
else:
    print(f"\nJob {status.status}. Check: client.fine_tuning.jobs.retrieve('{job_id}')")

## Applying LoRA in practice

This example mirrors the SFT workflow, but shows how LoRA is applied as a **parameter‑efficient method**. Data preparation is assumed to be identical to SFT. The important part is not the parameters. It is the deployment model: the adapter can be attached, detached, or replaced without disturbing the base model.

In [ ]:
%pip install -q peft transformers torch ipywidgets

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, PeftModel
import torch
import os
from pathlib import Path

BASE = "distilgpt2"  # small model -- runs on CPU

tokenizer = AutoTokenizer.from_pretrained(BASE)
tokenizer.pad_token = tokenizer.eos_token
base_model = AutoModelForCausalLM.from_pretrained(BASE)

# --- Configure LoRA adapter (rank, alpha, and which layers to target are explicit) ---
lora_config = LoraConfig(
    r=16,                          # low-rank dimension
    lora_alpha=32,                 # scaling factor
    target_modules=["c_attn"],     # inject into attention projection only
    lora_dropout=0.05,
    bias="none",
)

# ATTACH: wrap base model with LoRA -- base weights are frozen, only adapter params are trained
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

# --- Train adapter to learn one fact ---
FACT = "The capital of France is Paris."
train_enc = tokenizer(FACT, return_tensors="pt")
labels = train_enc["input_ids"].clone()

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
model.train()
for step in range(50):
    optimizer.zero_grad()
    loss = model(**train_enc, labels=labels).loss
    loss.backward()
    optimizer.step()
    if (step + 1) % 10 == 0:
        print(f"Step {step + 1}/50 -- loss: {loss.item():.4f}")
model.eval()

prompt = tokenizer("The capital of France is", return_tensors="pt")

# Stop at "." so generation ends at the sentence boundary
stop_token_id = tokenizer.encode(".", add_special_tokens=False)[0]
gen_kwargs = dict(max_new_tokens=10, eos_token_id=stop_token_id)

# ATTACH: adapter active -- model has learned the fact
print("\n=== Adapter ATTACHED ===")
with torch.no_grad():
    out = model.generate(**prompt, **gen_kwargs)
print(tokenizer.decode(out[0]))

# DETACH: bypass adapter -- base weights are untouched, original behaviour restored
model.disable_adapter_layers()
print("\n=== Adapter DETACHED (base model) ===")
with torch.no_grad():
    out = model.generate(**prompt, **gen_kwargs)
print(tokenizer.decode(out[0]))
model.enable_adapter_layers()

# SAVE: persist adapter to repo so Chapter 9 can load it for evaluation
adapter_dir = Path("../datasets/lora_adapter")
adapter_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(adapter_dir))
print(f"\nAdapter saved to {adapter_dir}")
print("Adapter files:", os.listdir(adapter_dir))
print(f"Total size: {sum(f.stat().st_size for f in adapter_dir.iterdir()) / 1024:.1f} KB")

# REPLACE: reload adapter onto a fresh base to verify it works
fresh_base = AutoModelForCausalLM.from_pretrained(BASE)
replaced_model = PeftModel.from_pretrained(fresh_base, str(adapter_dir))
replaced_model.eval()
print("\n=== Adapter REPLACED -- inference with reloaded adapter ===")
with torch.no_grad():
    out = replaced_model.generate(**prompt, **gen_kwargs)
print(tokenizer.decode(out[0]))

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 20500.81it/s]


trainable params: 294,912 || all params: 82,207,488 || trainable%: 0.3587
Step 10/50 — loss: 2.6535
Step 20/50 — loss: 1.9481
Step 30/50 — loss: 1.8068
Step 40/50 — loss: 0.4869


Setting `pad_token_id` to `eos_token_id`:13 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:13 for open-end generation.


Step 50/50 — loss: 0.0311

=== Adapter ATTACHED ===
The capital of France is Paris.

=== Adapter DETACHED (base model) ===
The capital of France is the capital of the French Republic.

Adapter saved to: <temp dir>
Adapter files: ['adapter_config.json', 'adapter_model.safetensors', 'README.md']


Loading weights: 100%|██████████| 76/76 [00:00<00:00, 15231.61it/s]
Setting `pad_token_id` to `eos_token_id`:13 for open-end generation.



=== Adapter REPLACED — inference with reloaded adapter ===
The capital of France is Paris.
